# ⚽ Scratchformer — FIFA World Cup Training (Day 7)

This notebook trains our from-scratch GPT model on the **FIFA World Cup** custom dataset using a **Colab T4 GPU**.

**What changed from Day 5 (Shakespeare)?**
- Dataset: FIFA World Cup history (1930–2026) instead of Tiny Shakespeare
- Vocab: ~79 characters (more digits, punctuation from stats/dates)
- Training: 7500 steps (larger corpus needs more time to converge)
- Warmup: 300 steps (new domain, gentler start helps)

**Workflow:**
1. Clone the latest code from GitHub
2. Install dependencies
3. Prepare the custom FIFA dataset (fetch + tokenize)
4. Train the model on T4 GPU with checkpoints saved to Google Drive
5. Plot loss curves and generate FIFA-themed sample text

---

### ⚡ Before you start
1. **Runtime → Change runtime type → T4 GPU**
2. Run the cells in order
3. Checkpoints are saved to Google Drive so you don't lose them if the Colab session disconnects

## 1. Setup — Clone Repo & Install Dependencies

In [ ]:
# Clone the latest code from GitHub
# This ensures Colab always has the most recent version of your .py files
!rm -rf scratchformer
!git clone https://github.com/aryannten/scratchformer.git
%cd scratchformer
!pip install -q -r requirements.txt

In [ ]:
# Mount Google Drive for persistent checkpoint storage
# Colab VMs are ephemeral — if the session dies, local files are gone.
# Drive is your safety net.
from google.colab import drive
drive.mount('/content/drive')

# Separate directory from Shakespeare checkpoints so they don't overwrite each other
DRIVE_CHECKPOINT_DIR = '/content/drive/MyDrive/scratchformer_checkpoints_fifa'
import os
os.makedirs(DRIVE_CHECKPOINT_DIR, exist_ok=True)
print(f'Checkpoints will be saved to: {DRIVE_CHECKPOINT_DIR}')

In [ ]:
# Verify GPU is available
import torch
print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available:  {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU:             {torch.cuda.get_device_name(0)}')
    total_mem = torch.cuda.mem_get_info(0)[1]
    print(f'Memory:          {total_mem / 1e9:.1f} GB')
else:
    print('⚠️  No GPU detected! Training will be very slow on CPU.')
    print('    Go to Runtime → Change runtime type → T4 GPU')

## 2. Prepare the FIFA World Cup Dataset

This fetches data from two sources:
1. **Historical CSVs** from [Fjelstul's worldcup repo](https://github.com/jfjelstul/worldcup) — tournaments, matches, goals, stadiums, awards, penalty shootouts, referees
2. **Wikipedia articles** — 2026 FIFA World Cup, qualification, venues

The raw CSV data is converted into **natural-language sentences** so the model learns English grammar alongside football facts.

In [ ]:
# Step 1: Fetch the FIFA World Cup data and build the corpus
# This creates data/raw/custom_corpus.txt
!python fetch_custom_data.py

In [ ]:
# Step 2: Tokenize and split into train/val
# This creates:
#   data/prepared/custom_train.pt   — 90% of the data, tokenized as integers
#   data/prepared/custom_val.pt     — 10% held out for evaluation
#   data/prepared/custom_vocab.json — the character vocabulary
!python prepare_data.py --dataset custom

In [ ]:
# Quick sanity check: load the data and inspect
from tokenizer import CharTokenizer
import json

tokenizer = CharTokenizer.load('data/prepared/custom_vocab.json')
train_data = torch.load('data/prepared/custom_train.pt', weights_only=True)
val_data = torch.load('data/prepared/custom_val.pt', weights_only=True)

print(f'Vocab size:    {tokenizer.vocab_size} characters')
print(f'Train tokens:  {len(train_data):,}')
print(f'Val tokens:    {len(val_data):,}')
print(f'\nSample text (first 500 chars):')
print(tokenizer.decode(train_data[:500].tolist()))

## 3. Pre-Training Sanity Check

Before spending GPU time on a full run, let's verify:
- Output shape is correct
- Initial loss is ~ln(vocab_size) ≈ ln(79) ≈ 4.37 (random guessing among 79 characters)
- Loss decreases after a few gradient steps

**Key difference from Shakespeare:** The FIFA dataset has a larger vocabulary (~79 vs 65 characters) because it includes digits, parentheses, and more punctuation from match statistics and dates.

In [ ]:
from model import Scratchformer, GPTConfig
from train import get_batch
import math

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Create model — vocab_size will be set from tokenizer
config = GPTConfig(vocab_size=tokenizer.vocab_size)
model = Scratchformer(config).to(device)

print(f'Model parameters: {model.count_parameters():,}')
print(f'Expected initial loss: {math.log(tokenizer.vocab_size):.4f}')

# Test forward pass
x, y = get_batch(train_data, batch_size=4, block_size=config.block_size, device=device)
logits, loss = model(x, y)

print(f'Output shape:    {logits.shape}  (expected: [4, {config.block_size}, {tokenizer.vocab_size}])')
print(f'Initial loss:    {loss.item():.4f}')
print(f'Loss is sane:    {"✅ Yes" if abs(loss.item() - math.log(tokenizer.vocab_size)) < 0.5 else "❌ No — investigate!"}')

In [ ]:
# Quick gradient step test: does loss actually decrease?
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

losses_check = []
for i in range(20):
    x, y = get_batch(train_data, batch_size=32, block_size=config.block_size, device=device)
    _, loss = model(x, y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    losses_check.append(loss.item())

print(f'Loss after  0 steps: {losses_check[0]:.4f}')
print(f'Loss after 20 steps: {losses_check[-1]:.4f}')
print(f'Loss decreased: {"✅ Yes — pipeline works!" if losses_check[-1] < losses_check[0] else "❌ No — something is wrong"}')

# Clean up — we'll create a fresh model for the real training
del model, optimizer
torch.cuda.empty_cache() if torch.cuda.is_available() else None

## 4. Train! ⚽🚀

Training the model on the FIFA World Cup dataset.

**Hyperparameter changes from Shakespeare:**

| Parameter | Shakespeare | FIFA | Why |
|---|---|---|---|
| `dataset` | `shakespeare` | `custom` | Different corpus |
| `max_steps` | 5000 | 7500 | Slightly larger corpus (~1.2MB vs ~1.1MB) + new domain needs more time |
| `warmup_steps` | 200 | 300 | New domain with more diverse patterns, gentler warmup helps |
| `eval_interval` | 250 | 250 | Same — frequent enough to catch problems |
| `save_interval` | 500 | 500 | Same — safety net for Colab disconnects |

On a T4 GPU, **7500 steps with batch_size=64 should take ~7–10 minutes**.

Checkpoints are saved:
- Every 500 steps → `checkpoints/step_XXXX.pt`
- Best validation loss → `checkpoints/best.pt`
- After training → `checkpoints/final.pt`

In [ ]:
from train import train, TrainConfig
from model import GPTConfig

# ── Model architecture ────────────────────────────────────
model_config = GPTConfig(
    # vocab_size is set automatically from the tokenizer
    block_size = 128,     # context window: 128 characters
    n_layer    = 4,       # 4 transformer blocks
    n_head     = 4,       # 4 attention heads
    n_embd     = 128,     # 128-dimensional embeddings
)

# ── Training hyperparameters ──────────────────────────────
# Key changes from Shakespeare:
#   - dataset='custom' to load the FIFA data
#   - max_steps=7500 for the slightly larger corpus + new domain
#   - warmup_steps=300 for a gentler start on unfamiliar data patterns
train_config = TrainConfig(
    dataset        = 'custom',
    max_steps      = 7500,        # more steps — new domain needs more time
    batch_size     = 64,          # same as Shakespeare
    learning_rate  = 3e-4,        # standard AdamW LR
    weight_decay   = 0.1,         # regularization
    grad_clip      = 1.0,         # gradient clipping
    warmup_steps   = 300,         # longer warmup for new domain
    eval_interval  = 250,         # eval every 250 steps
    save_interval  = 500,         # checkpoint every 500 steps
    checkpoint_dir = 'checkpoints',
)

# ── Launch training ───────────────────────────────────────
model, loss_log, tokenizer = train(
    model_config=model_config,
    train_config=train_config,
)

## 5. Results — Loss Curve

In [ ]:
# Display the loss curve
import matplotlib.pyplot as plt
from IPython.display import Image, display

# Re-plot inline for the notebook
fig, ax = plt.subplots(1, 1, figsize=(10, 6))

steps = [e['step'] for e in loss_log]
train_losses = [e['train'] for e in loss_log]
val_losses = [e['val'] for e in loss_log]

ax.plot(steps, train_losses, label='Train Loss', color='#4ECDC4', linewidth=2.5)
ax.plot(steps, val_losses, label='Val Loss', color='#FF6B6B', linewidth=2.5)
ax.set_xlabel('Step', fontsize=13)
ax.set_ylabel('Loss', fontsize=13)
ax.set_title('Scratchformer Training — FIFA World Cup', fontsize=15, fontweight='bold')
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)

# Annotate start and end
ax.annotate(f'Start: {train_losses[0]:.2f}', xy=(steps[0], train_losses[0]),
            fontsize=10, color='#4ECDC4', fontweight='bold',
            xytext=(steps[0]+300, train_losses[0]+0.1),
            arrowprops=dict(arrowstyle='->', color='#4ECDC4'))
ax.annotate(f'Final: {val_losses[-1]:.2f}', xy=(steps[-1], val_losses[-1]),
            fontsize=10, color='#FF6B6B', fontweight='bold',
            xytext=(steps[-1]-1200, val_losses[-1]+0.2),
            arrowprops=dict(arrowstyle='->', color='#FF6B6B'))

plt.tight_layout()
plt.show()

print(f'\nFinal train loss: {train_losses[-1]:.4f}')
print(f'Final val loss:   {val_losses[-1]:.4f}')

## 6. Generate Sample Text — FIFA Edition ⚽

Let's see what our trained model learned about football! We'll try prompts related to World Cup facts.

In [ ]:
def generate_text(model, tokenizer, prompt='', max_tokens=300, temperature=0.8, top_k=40):
    """Generate text from the trained model."""
    device = next(model.parameters()).device
    model.eval()

    if prompt:
        tokens = tokenizer.encode(prompt)
        idx = torch.tensor([tokens], dtype=torch.long, device=device)
    else:
        # Start with a newline character
        idx = torch.zeros((1, 1), dtype=torch.long, device=device)

    generated = model.generate(idx, max_new_tokens=max_tokens, temperature=temperature, top_k=top_k)
    return tokenizer.decode(generated[0].tolist())

In [ ]:
# Generation with different temperatures
print('=' * 60)
print('⚽ SAMPLE GENERATION — Temperature = 0.8 (balanced)')
print('=' * 60)
print(generate_text(model, tokenizer, temperature=0.8))

print('\n' + '=' * 60)
print('🧊 SAMPLE GENERATION — Temperature = 0.3 (conservative)')
print('=' * 60)
print(generate_text(model, tokenizer, temperature=0.3))

print('\n' + '=' * 60)
print('🔥 SAMPLE GENERATION — Temperature = 1.2 (creative)')
print('=' * 60)
print(generate_text(model, tokenizer, temperature=1.2))

In [ ]:
# Try FIFA-specific prompts
# These test whether the model learned real football knowledge
prompts = [
    'The 2022 FIFA World Cup',
    'Brazil won the',
    'In the final match',
    'The goal was scored by',
    'The 1930 World Cup was held in',
]

for prompt in prompts:
    print('=' * 60)
    print(f'📝 Prompt: "{prompt}"')
    print('=' * 60)
    output = generate_text(model, tokenizer, prompt=prompt, max_tokens=200, temperature=0.8)
    print(output)
    print()

## 6b. Shakespeare vs FIFA — Side-by-Side Comparison

Let's see how the same model architecture behaves differently with different training data.
Try the same generic prompt on both models to highlight the domain shift.

In [ ]:
# Compare: start from empty prompt at temp=0.8
# The FIFA model should produce football-related text
# while the Shakespeare model (Day 5) produced dialogue with character names

print('=' * 60)
print('⚽ FIFA MODEL — Empty prompt, temperature=0.8')
print('=' * 60)
for i in range(3):
    print(f'\n--- Sample {i+1} ---')
    print(generate_text(model, tokenizer, temperature=0.8, max_tokens=150))

## 7. Copy Checkpoints to Google Drive

Save the best and final checkpoints to Drive so they persist even after the Colab session ends.

**Note:** These go to `scratchformer_checkpoints_fifa/` to keep them separate from the Shakespeare checkpoints.

In [ ]:
import shutil

# Copy key checkpoints to Google Drive
for ckpt_name in ['best.pt', 'final.pt', 'loss_curve.png']:
    src = f'checkpoints/{ckpt_name}'
    dst = f'{DRIVE_CHECKPOINT_DIR}/{ckpt_name}'
    if os.path.exists(src):
        shutil.copy2(src, dst)
        print(f'✅ Copied {ckpt_name} → {dst}')
    else:
        print(f'⚠️  {src} not found, skipping')

# Also save the custom vocab so we can decode later
shutil.copy2('data/prepared/custom_vocab.json', f'{DRIVE_CHECKPOINT_DIR}/custom_vocab.json')
print(f'✅ Copied custom_vocab.json → {DRIVE_CHECKPOINT_DIR}/custom_vocab.json')

print(f'\n📁 Drive contents:')
for f in os.listdir(DRIVE_CHECKPOINT_DIR):
    size = os.path.getsize(os.path.join(DRIVE_CHECKPOINT_DIR, f))
    print(f'   {f:30s} {size / 1e6:.1f} MB')

## 8. What to Look For

### Loss curve health check:
- ✅ **Both lines trend downward** — learning is happening
- ✅ **Train and val track each other** — not overfitting (yet)
- ⚠️ **Val loss flattens while train drops** — beginning to overfit (might happen earlier than Shakespeare since FIFA corpus has more repetitive structure)
- ❌ **Loss is spiky or increasing** — LR too high, try reducing to 1e-4

### Expected initial loss:
- **Random init**: ~4.37 (ln(79)) — slightly higher than Shakespeare's 4.17 because of the larger vocabulary
- **After 7500 steps**: ~1.3–1.8 is a good target
- The FIFA dataset has more structured/repetitive patterns (dates, scores, "scored by") so the model might converge to a lower loss than Shakespeare

### Generation quality checklist:
- [ ] Real country names appear (Brazil, Germany, Argentina, etc.)
- [ ] Year-like patterns appear (1930, 1954, 2022, etc.)
- [ ] Football-specific vocabulary (goal, match, stadium, tournament, etc.)
- [ ] Sentence structure resembles the training data format
- [ ] Scores and statistics look plausible (even if factually wrong)

### What makes FIFA different from Shakespeare:
- More **factual, structured** text vs. **creative, poetic** text
- More **numbers and dates** in the vocabulary
- More **repetitive sentence patterns** ("X scored a goal in minute Y") — the model should learn these templates faster
- Less **long-range dependency** — facts are mostly self-contained sentences, not multi-page dialogue

---

### ⏭️ Next: Day 8
If the generation looks coherent:
1. Try tweaking hyperparameters (more steps, different LR, larger model)
2. Compare Shakespeare vs FIFA outputs side-by-side
3. Pick the best checkpoint for the Gradio demo